# Inconsistent metal connectivity in Bond objects and `madjmat`

This notebook documents an unintended connectivity procedure identified in SCOPE 0.9.5. Metal--ligand `Bond` objects can disagree with the molecule's authoritative metal-adjacency matrix (`madjmat`) because they are generated using a separate connectivity calculation with default parameters.

The discrepancy was found in WOJLUC. A nitrogen at 2.5038 Angstrom from Fe is absent from both `adjmat` and `madjmat`, but appears as directly Fe-bound in `Molecule.get_graph()` because the graph uses the stored `Bond` objects when `has_bonds=True`. No System is modified by this notebook.

In [1]:
import os

import numpy as np
import scope

from scope.connectivity import get_adjmatrix, get_radii
from scope.operations.graphs import compute_topological_distances

## Part 1. Synthetic threshold mismatch

`Molecule.get_metal_adjmatrix()` uses the molecule-specific `cov_factor`, `metal_factor`, and radii. In contrast, `Molecule.set_metal_ligand_bonds()` calls `Atom.check_connectivity()`, which recomputes connectivity for each atom pair using the default factors. The following Fe--N pair reproduces the WOJLUC boundary case.

In [2]:
labels = ['Fe', 'N']
distance = 2.5038
coordinates = [[0.0, 0.0, 0.0], [distance, 0.0, 0.0]]
radii = get_radii(labels)

molecule_cov_factor = 1.36
molecule_metal_factor = 0.86

_, molecule_madjmat, _ = get_adjmatrix(
    labels,
    coordinates,
    cov_factor=molecule_cov_factor,
    metal_factor=molecule_metal_factor,
    radii=radii,
    metal_only=True,
)
_, pair_adjmat, _ = get_adjmatrix(labels, coordinates)

molecule_threshold = molecule_cov_factor * (
    radii[0] * molecule_metal_factor + radii[1]
)
pair_default_threshold = 1.3 * sum(radii)

print('Fe-N distance:', distance)
print('Molecule-specific threshold:', molecule_threshold)
print('Default pair threshold:', pair_default_threshold)
print('Connected in molecule madjmat:', bool(molecule_madjmat[0, 1]))
print('Connected by default pair test:', bool(pair_adjmat[0, 1]))

Fe-N distance: 2.5038
Molecule-specific threshold: 2.4920640000000005
Default pair threshold: 2.6260000000000003
Connected in molecule madjmat: False
Connected by default pair test: True


The expected output is `False` for the molecule-specific `madjmat` and `True` for the default pair test. Therefore, `set_metal_ligand_bonds()` can create a metal--ligand `Bond` that is inconsistent with `madjmat` from the moment it is constructed; temporal cache staleness is not required.

There is a separate cache-staleness risk because rebuilding bonds does not first clear all existing per-atom bond lists. Obsolete bonds may therefore also survive a coordinate or connectivity update.

## Part 2. WOJLUC

This optional section reproduces the original dataset case when the SCO ATLAS System file is available.

In [3]:
wojluc_path = '/Users/sergivela/Documents/SCOPE/Database_SCO/Data/2-Systems/WOJLUC/WOJLUC.npy'

if os.path.isfile(wojluc_path):
    system = scope.load_binary(wojluc_path)
    found, molecule = system.find_source('ref_hs_mol')
    assert found

    ligand = molecule.ligands[0]
    ligand_indices = ligand.get_parent_indices('molecule')
    metal_index = molecule.metals[0].get_parent_index('molecule')
    graph = molecule.get_graph()
    topological_distances = compute_topological_distances(graph, metal_index)
    metal_coordinates = np.asarray(molecule.atoms[metal_index].coord, dtype=float)

    print('Ligand denticity:', ligand.get_denticity())
    print('Molecule factors:', molecule.cov_factor, molecule.metal_factor)
    print('N atoms classified as directly Fe-bound by graph or madjmat:')
    for ligand_index, molecule_index in enumerate(ligand_indices):
        if molecule.labels[molecule_index] != 'N':
            continue
        graph_connected = int(topological_distances[molecule_index]) == 1
        matrix_connected = bool(molecule.madjmat[metal_index, molecule_index])
        if graph_connected or matrix_connected:
            fe_n_distance = np.linalg.norm(
                np.asarray(molecule.atoms[molecule_index].coord, dtype=float)
                - metal_coordinates
            )
            print({
                'ligand_index': ligand_index,
                'molecule_index': molecule_index,
                'Fe-N distance [A]': round(float(fe_n_distance), 4),
                'graph_connected': graph_connected,
                'madjmat_connected': matrix_connected,
            })
else:
    print('WOJLUC System not found:', wojluc_path)

Ligand denticity: 6
Molecule factors: 1.36 0.86
N atoms classified as directly Fe-bound by graph or madjmat:
{'ligand_index': 0, 'molecule_index': 1, 'Fe-N distance [A]': 2.5038, 'graph_connected': True, 'madjmat_connected': False}
{'ligand_index': 17, 'molecule_index': 18, 'Fe-N distance [A]': 2.2272, 'graph_connected': True, 'madjmat_connected': True}
{'ligand_index': 18, 'molecule_index': 19, 'Fe-N distance [A]': 2.2272, 'graph_connected': True, 'madjmat_connected': True}
{'ligand_index': 19, 'molecule_index': 20, 'Fe-N distance [A]': 2.2272, 'graph_connected': True, 'madjmat_connected': True}
{'ligand_index': 33, 'molecule_index': 34, 'Fe-N distance [A]': 2.3265, 'graph_connected': True, 'madjmat_connected': True}
{'ligand_index': 34, 'molecule_index': 35, 'Fe-N distance [A]': 2.3265, 'graph_connected': True, 'madjmat_connected': True}
{'ligand_index': 35, 'molecule_index': 36, 'Fe-N distance [A]': 2.3265, 'graph_connected': True, 'madjmat_connected': True}


For WOJLUC, six N atoms are connected according to both representations. A seventh N at approximately 2.5038 Angstrom is connected only in the graph. This makes the graph-derived coordination number seven even though `ligand.get_denticity()` and `molecule.madjmat` correctly indicate a hexadentate ligand.

## Suggested correction

1. Construct metal--ligand `Bond` objects directly from the molecule's current `madjmat`, rather than recomputing pair connectivity with default parameters.
2. Clear existing bond lists before rebuilding `Bond` objects.
3. When constructing a molecular graph, preserve covalent bond-order information from `Bond` objects but synchronize all metal edges with `madjmat`.

The invariant to preserve is that direct metal neighbours in `Molecule.get_graph()` must equal the nonzero entries in the corresponding row of `Molecule.madjmat`.

In [4]:
# Enable after applying the correction to use WOJLUC as a regression test.
test_fix = False

if test_fix and os.path.isfile(wojluc_path):
    graph_neighbours = set(graph.neighbors(metal_index))
    matrix_neighbours = set(np.flatnonzero(molecule.madjmat[metal_index]))
    assert graph_neighbours == matrix_neighbours
    assert len(graph_neighbours) == 6
    print('Metal graph and madjmat are consistent')